In [ ]:
# STEP 1: Setup — install and connect DuckDB 
!pip install duckdb -q

import duckdb
con = duckdb.connect()
con.execute("""
    CREATE TABLE customers AS
    SELECT * FROM read_csv('customer_features_engineered.csv')
""")

# Sanity check
print(con.execute("SELECT COUNT(*) AS row_count FROM customers").df())
print(con.execute("PRAGMA table_info(customers)").df()[['name']])


# Q1. What separates high-value customers from low-value ones, and which
#     profiles show the strongest repeat purchase behavior?
query = """
SELECT
    "Value_Tier",
    COUNT(*) AS customer_count,
    ROUND(AVG("Estimated_Lifetime_Spend"), 0) AS avg_lifetime_spend,
    ROUND(AVG("Previous Purchases"), 1) AS avg_previous_purchases,
    ROUND(AVG("Review Rating"), 2) AS avg_review_rating,
    ROUND(AVG("Promo_Dependency_Score"), 3) AS avg_promo_dependency,
    ROUND(100.0 * SUM(CASE WHEN "Engagement_Depth_Tier" = 'Veteran'
                            THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_veteran_tenure
FROM customers
GROUP BY "Value_Tier"
ORDER BY avg_lifetime_spend DESC;
"""
con.execute(query).df()


# Q2a. Which categories are associated with lower-tenure vs high-tenure customers?
query = """
SELECT
    "Category",
    COUNT(*) AS customer_count,
    ROUND(AVG("Previous Purchases"), 1) AS avg_previous_purchases,
    ROUND(100.0 * SUM(CASE WHEN "Engagement_Depth_Tier" = 'New'
                            THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_new_customers,
    ROUND(100.0 * SUM(CASE WHEN "Engagement_Depth_Tier" = 'Veteran'
                            THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_veteran_customers
FROM customers
GROUP BY "Category"
ORDER BY avg_previous_purchases DESC;
"""
con.execute(query).df()


# Q2b. Which seasons are associated with lower-tenure vs high-tenure customers?
query = """
SELECT
    "Season",
    COUNT(*) AS customer_count,
    ROUND(AVG("Previous Purchases"), 1) AS avg_previous_purchases,
    ROUND(100.0 * SUM(CASE WHEN "Engagement_Depth_Tier" = 'Veteran'
                            THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_veteran_customers
FROM customers
GROUP BY "Season"
ORDER BY avg_previous_purchases DESC;
"""
con.execute(query).df()


# Q3a. Which geographies (region-level) signal organic demand vs discount-driven volume?
query = """
SELECT
    "Region",
    COUNT(*) AS customer_count,
    ROUND(AVG("Estimated_Lifetime_Spend"), 0) AS avg_lifetime_spend,
    ROUND(AVG("Promo_Dependency_Score"), 3) AS avg_promo_dependency,
    ROUND(AVG("Loyalty_Score_A_Behavioral"), 3) AS avg_engagement_depth
FROM customers
GROUP BY "Region"
ORDER BY avg_lifetime_spend DESC;
"""
con.execute(query).df()


# Q3b. State-level drill-down: high spend + low promo dependency (min. 60 customers per state for statistical stability)
query = """
SELECT
    "Location",
    COUNT(*) AS customer_count,
    ROUND(AVG("Estimated_Lifetime_Spend"), 0) AS avg_lifetime_spend,
    ROUND(AVG("Promo_Dependency_Score"), 3) AS avg_promo_dependency
FROM customers
GROUP BY "Location"
HAVING COUNT(*) >= 60
ORDER BY avg_lifetime_spend DESC, avg_promo_dependency ASC
LIMIT 10;
"""
con.execute(query).df()


In [ ]:
Q4. Who are the genuinely loyal customers vs. those who only buy when there's a discount?
query = """
SELECT
    "Customer_Segment",
    COUNT(*) AS customer_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customers), 1) AS pct_of_base,
    ROUND(AVG("Estimated_Lifetime_Spend"), 0) AS avg_lifetime_spend,
    ROUND(AVG("Promo_Dependency_Score"), 3) AS avg_promo_dependency,
    ROUND(100.0 * SUM("Discount Applied_Flag") / COUNT(*), 1) AS pct_using_discount,
    ROUND(AVG("Review Rating"), 2) AS avg_review_rating
FROM customers
GROUP BY "Customer_Segment"
ORDER BY avg_lifetime_spend DESC;
"""
con.execute(query).df()


# Q5. What behavioral patterns today predict high customer value?
#     (Engagement Depth Tier x Value Tier cross-tab)
query = """
SELECT
    "Engagement_Depth_Tier",
    "Value_Tier",
    COUNT(*) AS customer_count
FROM customers
GROUP BY "Engagement_Depth_Tier", "Value_Tier"
ORDER BY "Engagement_Depth_Tier", "Value_Tier";
"""
con.execute(query).df()


# Q6. Which demographics (age bands) are commercially underserved?
query = """
SELECT
    CASE
        WHEN "Age" < 25 THEN '18-24'
        WHEN "Age" < 35 THEN '25-34'
        WHEN "Age" < 45 THEN '35-44'
        WHEN "Age" < 55 THEN '45-54'
        WHEN "Age" < 65 THEN '55-64'
        ELSE '65+'
    END AS age_band,
    COUNT(*) AS customer_count,
    ROUND(AVG("Estimated_Lifetime_Spend"), 0) AS avg_lifetime_spend,
    ROUND(100.0 * SUM(CASE WHEN "Customer_Segment" = 'Core Loyal'
                            THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_core_loyal
FROM customers
GROUP BY age_band
ORDER BY avg_lifetime_spend DESC;
"""
con.execute(query).df()


# Q7. How should the brand restructure its promotional strategy?
#     (promo dependency by segment — basis for the sunset plan)
query = """
SELECT
    "Customer_Segment",
    COUNT(*) AS customer_count,
    ROUND(AVG("Promo_Dependency_Score"), 3) AS avg_promo_dependency,
    ROUND(100.0 * SUM("Discount Applied_Flag") / COUNT(*), 1) AS pct_using_discount,
    ROUND(AVG("Estimated_Lifetime_Spend"), 0) AS avg_lifetime_spend
FROM customers
GROUP BY "Customer_Segment"
ORDER BY avg_promo_dependency DESC;
"""
con.execute(query).df()


# Q8a. Ideal Customer Profile — Core Loyal segment summary stats
query = """
SELECT
    COUNT(*) AS n_core_loyal,
    ROUND(AVG("Age"), 1) AS avg_age,
    ROUND(AVG("Estimated_Lifetime_Spend"), 0) AS avg_lifetime_spend,
    ROUND(AVG("Review Rating"), 2) AS avg_review_rating,
    ROUND(AVG("Previous Purchases"), 1) AS avg_previous_purchases
FROM customers
WHERE "Customer_Segment" = 'Core Loyal';
"""
con.execute(query).df()


# Q8b. Ideal Customer Profile — top category within Core Loyal
query = """
SELECT "Category", COUNT(*) AS n
FROM customers
WHERE "Customer_Segment" = 'Core Loyal'
GROUP BY "Category"
ORDER BY n DESC;
"""
con.execute(query).df()


# Q8c. Ideal Customer Profile — top payment method within Core Loyal
query = """
SELECT "Payment Method", COUNT(*) AS n
FROM customers
WHERE "Customer_Segment" = 'Core Loyal'
GROUP BY "Payment Method"
ORDER BY n DESC;
"""
con.execute(query).df()


# Q8d. Ideal Customer Profile — gender split within Core Loyal
# (compare against overall dataset ratio to check if it's a real skew)
query = """
SELECT "Gender", COUNT(*) AS n
FROM customers
WHERE "Customer_Segment" = 'Core Loyal'
GROUP BY "Gender"
ORDER BY n DESC;
"""
con.execute(query).df()


# Q8e. Ideal Customer Profile — shipping type preference within Core Loyal
query = """
SELECT "Shipping Type", COUNT(*) AS n
FROM customers
WHERE "Customer_Segment" = 'Core Loyal'
GROUP BY "Shipping Type"
ORDER BY n DESC;
"""
con.execute(query).df()